# Few-shot baseline

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT


import gc
import json
import random
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "surprise", "sadness"]

MAIN_CSV = "data/combined_emotions.csv"

BATCH_SIZE = 64
MAX_LENGTH = 1024
SEED = 42

OUTPUT_DIR = "data/few_shot_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Using device: {DEVICE}")


# ============================================================
# Helpers
# ============================================================

def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def zero_shot_template(text: str) -> str:
    return f"In this text: {text}, the emotion implied is:"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_emotion_token_maps(tokenizer, emotions: List[str]):
    emotion_to_token = {}
    token_to_emotion = {}

    print("Emotion target tokens:")

    for emotion in emotions:
        token_ids = tokenizer.encode(" " + emotion)
        token_id = int(token_ids[-1])

        emotion_to_token[emotion] = token_id
        token_to_emotion[token_id] = emotion

        pieces = [tokenizer.decode([int(x)]) for x in token_ids]
        print(
            f"  {emotion}: ids={token_ids}, pieces={pieces}, "
            f"using={token_id} ({tokenizer.decode([token_id])!r})"
        )

    if len(set(emotion_to_token.values())) != len(emotions):
        raise ValueError(f"Duplicate target token IDs found: {emotion_to_token}")

    return emotion_to_token, token_to_emotion


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=None,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    # Important for few-shot prompts:
    # keep the final query and final "Emotion:" cue
    tokenizer.truncation_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=DEVICE,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        cache_dir=None,
    )

    model.eval()
    return model, tokenizer


def get_final_token_logits(outputs, attention_mask):
    """
    Robust final-token indexing for left or right padding.
    """
    logits = outputs.logits
    mask = attention_mask.long()
    positions = torch.arange(mask.size(1), device=mask.device).unsqueeze(0)
    final_indices = (mask * positions).max(dim=1).values
    batch_indices = torch.arange(logits.size(0), device=logits.device)
    return logits[batch_indices, final_indices, :]


# ============================================================
# Dataset loading
# ============================================================

def load_main_train_test():
    train_df = pd.read_csv("data/train_df.csv")
    test_df = pd.read_csv("data/test_df.csv")
    if set(train_df.input_text) & set(test_df.input_text):
        raise ValueError("Few-shot demonstrations overlap test sentences")
    return train_df, test_df


def load_xed():
    df = pd.read_csv("data/xed-en.csv")

    df = df[["text", "emotion"]].dropna().copy()
    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    df = df[df["emotion"].isin(EMOTIONS)].copy()

    df["input_text"] = df["text"].apply(zero_shot_template)

    return df[["input_text", "emotion"]].reset_index(drop=True)


def load_emoevent():
    df = pd.read_csv("data/emoevent_en.csv", sep="\t")

    df = df[df["emotion"] != "others"].copy()
    df = df[["tweet", "emotion"]].dropna().copy()

    df["emotion"] = df["emotion"].astype(str).str.lower().str.strip()
    df = df[df["emotion"].isin(EMOTIONS)].copy()

    df["input_text"] = df["tweet"].apply(zero_shot_template)

    return df[["input_text", "emotion"]].reset_index(drop=True)


# ============================================================
# Few-shot prompt builder
# ============================================================

def build_train_by_emotion(train_df: pd.DataFrame):
    train_by_emotion = {
        emotion: train_df[train_df["emotion"] == emotion].reset_index(drop=True)
        for emotion in EMOTIONS
    }

    for emotion in EMOTIONS:
        if len(train_by_emotion[emotion]) == 0:
            raise ValueError(f"No training examples found for emotion: {emotion}")

    return train_by_emotion


def build_few_shot_prompt(
    query_input_prompt: str,
    train_by_emotion: Dict[str, pd.DataFrame],
    rng: np.random.Generator,
):
    examples = []

    for emotion in EMOTIONS:
        emo_df = train_by_emotion[emotion]
        idx = rng.integers(len(emo_df))

        example_prompt = emo_df.iloc[idx]["input_text"]

        examples.append(
            f"{example_prompt}\nEmotion: {emotion}"
        )

    few_shot_block = "\n\n".join(examples)

    return few_shot_block + "\n\n" + f"{query_input_prompt}\nEmotion:"


def add_few_shot_prompts(
    test_df: pd.DataFrame,
    train_by_emotion: Dict[str, pd.DataFrame],
    seed: int,
):
    rng = np.random.default_rng(seed)

    out = test_df.copy()
    out["few_shot_prompt"] = out["input_text"].apply(
        lambda x: build_few_shot_prompt(
            query_input_prompt=x,
            train_by_emotion=train_by_emotion,
            rng=rng,
        )
    )

    return out


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_few_shot_dataset(
    model,
    tokenizer,
    df: pd.DataFrame,
    dataset_name: str,
    model_name: str,
    emotion_to_token: Dict[str, int],
    batch_size: int = 32,
    max_length: int = 512,
):
    target_token_ids = [emotion_to_token[e] for e in EMOTIONS]

    all_target_logits = []

    input_texts = df["few_shot_prompt"].tolist()

    for start in tqdm(
        range(0, len(input_texts), batch_size),
        desc=f"{model_safe_name(model_name)} | {dataset_name}",
    ):
        batch_texts = input_texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        input_ids = inputs["input_ids"].to(DEVICE)
        attention_mask = inputs["attention_mask"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        final_logits = get_final_token_logits(outputs, attention_mask)
        batch_target_logits = final_logits[:, target_token_ids]

        all_target_logits.append(batch_target_logits.detach().float().cpu())

        del input_ids, attention_mask, outputs, final_logits, batch_target_logits
        cleanup_memory()

    all_target_logits = torch.cat(all_target_logits, dim=0).numpy()

    pred_indices = np.argmax(all_target_logits, axis=1)
    y_pred = [EMOTIONS[i] for i in pred_indices]
    y_true = df["emotion"].astype(str).str.lower().str.strip().tolist()

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=EMOTIONS,
        output_dict=True,
        zero_division=0,
    )

    report_str = classification_report(
        y_true,
        y_pred,
        labels=EMOTIONS,
        zero_division=0,
    )

    summary = {
        "model_name": model_name,
        "model_safe_name": model_safe_name(model_name),
        "dataset": dataset_name,
        "n": int(len(df)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=EMOTIONS, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=EMOTIONS, average="weighted", zero_division=0)),
    }

    per_emotion_rows = []

    for emotion in EMOTIONS:
        r = report_dict.get(emotion, {})
        per_emotion_rows.append({
            "model_name": model_name,
            "model_safe_name": model_safe_name(model_name),
            "dataset": dataset_name,
            "emotion": emotion,
            "precision": float(r.get("precision", 0.0)),
            "recall": float(r.get("recall", 0.0)),
            "f1": float(r.get("f1-score", 0.0)),
            "support": int(r.get("support", 0)),
        })

    pred_df = df.copy()
    pred_df["pred_emotion"] = y_pred

    for j, emotion in enumerate(EMOTIONS):
        pred_df[f"logit_{emotion}"] = all_target_logits[:, j]

    print("\n" + "=" * 80)
    print(f"{model_name} | {dataset_name}")
    print("=" * 80)
    print(report_str)

    return summary, pd.DataFrame(per_emotion_rows), pred_df, report_dict, report_str


# ============================================================
# Save helpers
# ============================================================

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def run_one_model(
    model_name: str,
    train_df: pd.DataFrame,
    datasets: Dict[str, pd.DataFrame],
):
    set_seed(SEED)

    safe_name = model_safe_name(model_name)
    model_output_dir = os.path.join(OUTPUT_DIR, safe_name)
    os.makedirs(model_output_dir, exist_ok=True)

    model, tokenizer = load_model_and_tokenizer(model_name)

    emotion_to_token, token_to_emotion = build_emotion_token_maps(
        tokenizer=tokenizer,
        emotions=EMOTIONS,
    )

    save_json(
        {
            "model_name": model_name,
            "emotion_to_token": emotion_to_token,
            "token_to_emotion": {str(k): v for k, v in token_to_emotion.items()},
        },
        os.path.join(model_output_dir, "emotion_token_map.json"),
    )

    train_by_emotion = build_train_by_emotion(train_df)

    summary_rows = []
    per_emotion_dfs = []

    for dataset_name, raw_df in datasets.items():
        fewshot_df = add_few_shot_prompts(
            test_df=raw_df,
            train_by_emotion=train_by_emotion,
            seed=SEED,
        )

        summary, per_emotion_df, pred_df, report_dict, report_str = evaluate_few_shot_dataset(
            model=model,
            tokenizer=tokenizer,
            df=fewshot_df,
            dataset_name=dataset_name,
            model_name=model_name,
            emotion_to_token=emotion_to_token,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
        )

        summary_rows.append(summary)
        per_emotion_dfs.append(per_emotion_df)

        dataset_dir = os.path.join(model_output_dir, dataset_name)
        os.makedirs(dataset_dir, exist_ok=True)

        pred_df.to_csv(
            os.path.join(dataset_dir, "predictions.csv"),
            index=False,
        )

        pd.DataFrame([summary]).to_csv(
            os.path.join(dataset_dir, "summary.csv"),
            index=False,
        )

        per_emotion_df.to_csv(
            os.path.join(dataset_dir, "per_emotion_f1.csv"),
            index=False,
        )

        save_json(
            report_dict,
            os.path.join(dataset_dir, "classification_report.json"),
        )

        with open(os.path.join(dataset_dir, "classification_report.txt"), "w") as f:
            f.write(report_str)

    model_summary_df = pd.DataFrame(summary_rows)
    model_per_emotion_df = pd.concat(per_emotion_dfs, ignore_index=True)

    model_summary_df.to_csv(
        os.path.join(model_output_dir, "all_dataset_summary.csv"),
        index=False,
    )

    model_per_emotion_df.to_csv(
        os.path.join(model_output_dir, "all_dataset_per_emotion_f1.csv"),
        index=False,
    )

    del model
    del tokenizer
    cleanup_memory()

    return model_summary_df, model_per_emotion_df

In [ ]:
# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)

    train_df, main_test_df = load_main_train_test()

    datasets = {
        "main_test": main_test_df,
        "xed_en": load_xed(),
        "emoevent_en": load_emoevent(),
    }

    print("\nDataset sizes:")
    for name, df in datasets.items():
        print(name, len(df), df["emotion"].value_counts().to_dict())

    all_summary_dfs = []
    all_per_emotion_dfs = []

    failures = []

    for model_name in MODELS:
        try:
            print("\n" + "#" * 100)
            print(f"Running few-shot evaluation for: {model_name}")
            print("#" * 100)

            summary_df, per_emotion_df = run_one_model(
                model_name=model_name,
                train_df=train_df,
                datasets=datasets,
            )

            all_summary_dfs.append(summary_df)
            all_per_emotion_dfs.append(per_emotion_df)

        except Exception as e:
            print(f"FAILED: {model_name} | {repr(e)}")
            failures.append({
                "model_name": model_name,
                "error": repr(e),
            })
            cleanup_memory()

    if all_summary_dfs:
        all_summary = pd.concat(all_summary_dfs, ignore_index=True)
        all_summary.to_csv(
            os.path.join(OUTPUT_DIR, "all_models_few_shot_summary.csv"),
            index=False,
        )

    if all_per_emotion_dfs:
        all_per_emotion = pd.concat(all_per_emotion_dfs, ignore_index=True)
        all_per_emotion.to_csv(
            os.path.join(OUTPUT_DIR, "all_models_few_shot_per_emotion_f1.csv"),
            index=False,
        )

    if failures:
        pd.DataFrame(failures).to_csv(
            os.path.join(OUTPUT_DIR, "failures.csv"),
            index=False,
        )
        print("\nFailures:")
        print(pd.DataFrame(failures))

    print("\nDone.")
    print(f"Saved results to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import pandas as pd
import numpy as np


# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

FEW_SHOT_RESULT_DIR = "data/few_shot_results"

OUTPUT_CSV = "data/few_shot_results/all_models_few_shot_table_results.csv"


def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def display_model_name(model_name: str) -> str:
    name_map = {
        "google/gemma-2-2b": "Gemma-2-2B",
        "google/gemma-2-9b": "Gemma-2-9B",
        "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
        "google/gemma-3-4b-it": "Gemma-3-4B-IT",
        "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
        "google/gemma-3-12b-it": "Gemma-3-12B-IT",
        "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
    }
    return name_map.get(model_name, model_name)


def normalize_dataset_name(x: str) -> str:
    x = str(x).lower().strip()

    if x in {
        "main",
        "main_test",
        "self_reported",
        "self-reported",
        "self_reported_ekman",
    }:
        return "main"

    if x in {
        "emoevent",
        "emoevent_en",
        "emo_event",
        "emo_event_en",
    }:
        return "emoevent"

    if x in {
        "xed",
        "xed_en",
        "xed-en",
    }:
        return "xed"

    return x


def read_model_few_shot_results(model_name: str):
    safe_name = model_safe_name(model_name)
    model_dir = os.path.join(FEW_SHOT_RESULT_DIR, safe_name)

    summary_path = os.path.join(model_dir, "all_dataset_summary.csv")
    per_emotion_path = os.path.join(model_dir, "all_dataset_per_emotion_f1.csv")

    if not os.path.exists(summary_path):
        print(f"[Missing] {summary_path}")
        return None

    if not os.path.exists(per_emotion_path):
        print(f"[Missing] {per_emotion_path}")
        return None

    summary_df = pd.read_csv(summary_path)
    per_emotion_df = pd.read_csv(per_emotion_path)

    summary_df["dataset_norm"] = summary_df["dataset"].apply(normalize_dataset_name)
    per_emotion_df["dataset_norm"] = per_emotion_df["dataset"].apply(normalize_dataset_name)

    row = {
        "model": display_model_name(model_name),
        "setting": "Few-shot",
        "main_macro_f1": np.nan,
        "anger_f1": np.nan,
        "disgust_f1": np.nan,
        "fear_f1": np.nan,
        "joy_f1": np.nan,
        "sadness_f1": np.nan,
        "surprise_f1": np.nan,
        "emoevent_macro_f1": np.nan,
        "xed_macro_f1": np.nan,
        "ppl": "--",
    }

    # ------------------------------------------------------------
    # Main macro-F1
    # ------------------------------------------------------------
    main_summary = summary_df[summary_df["dataset_norm"] == "main"]

    if len(main_summary) > 0:
        row["main_macro_f1"] = float(main_summary.iloc[0]["macro_f1"])
    else:
        print(f"[Warning] No main dataset summary for {model_name}. Available:")
        print(summary_df["dataset"].unique())

    # ------------------------------------------------------------
    # External macro-F1
    # ------------------------------------------------------------
    emoevent_summary = summary_df[summary_df["dataset_norm"] == "emoevent"]
    if len(emoevent_summary) > 0:
        row["emoevent_macro_f1"] = float(emoevent_summary.iloc[0]["macro_f1"])

    xed_summary = summary_df[summary_df["dataset_norm"] == "xed"]
    if len(xed_summary) > 0:
        row["xed_macro_f1"] = float(xed_summary.iloc[0]["macro_f1"])

    # ------------------------------------------------------------
    # Main per-emotion F1
    # ------------------------------------------------------------
    main_per_emotion = per_emotion_df[per_emotion_df["dataset_norm"] == "main"].copy()

    if len(main_per_emotion) > 0:
        for emotion in EMOTIONS:
            emo_row = main_per_emotion[main_per_emotion["emotion"] == emotion]
            if len(emo_row) > 0:
                row[f"{emotion}_f1"] = float(emo_row.iloc[0]["f1"])
            else:
                print(f"[Warning] Missing {emotion} F1 for {model_name}")
    else:
        print(f"[Warning] No main per-emotion F1 for {model_name}. Available:")
        print(per_emotion_df["dataset"].unique())

    return row


# ============================================================
# Build combined table
# ============================================================

rows = []

for model_name in MODELS:
    row = read_model_few_shot_results(model_name)
    if row is not None:
        rows.append(row)

fewshot_table_df = pd.DataFrame(rows)

# Desired column order
cols = [
    "model",
    "setting",
    "main_macro_f1",
    "anger_f1",
    "disgust_f1",
    "fear_f1",
    "joy_f1",
    "sadness_f1",
    "surprise_f1",
    "emoevent_macro_f1",
    "xed_macro_f1",
    "ppl",
]

fewshot_table_df = fewshot_table_df[cols]

# Round numeric columns
for col in fewshot_table_df.columns:
    if col not in {"model", "setting", "ppl"}:
        fewshot_table_df[col] = pd.to_numeric(fewshot_table_df[col], errors="coerce").round(3)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
fewshot_table_df.to_csv(OUTPUT_CSV, index=False)

print("Saved to:", OUTPUT_CSV)
display(fewshot_table_df)